# Voice Embedding Training (ECAPA-TDNN speaker verification)

Fine-tunes the voice embedding model used by `models/voice/inference.py` (SpeechBrain ECAPA-TDNN + ArcFace, see `docs/VOICE_MODEL.md`), evaluates it (precision/recall/F1/FAR/FRR/EER/ROC-AUC), and runs a genuine-vs-impostor demo.

**Dataset:** [Voxceleb1 audio wav files for India celebrity](https://www.kaggle.com/datasets/gaurav41/voxceleb1-audio-wav-files-for-india-celebrity) - a real-audio subset of VoxCeleb1 (Indian-celebrity speakers), DbCL-1.0 licensed. This is a subset, not the full VoxCeleb1 corpus - see `docs/DATASETS.md`'s Voice section for why a full-corpus mirror wasn't used and what that means for the numbers this notebook reports.

## 1. Setup

In [ ]:
# --- Kaggle Kernel setup ---
# GPU + internet are enabled in kernel-metadata.json; the dataset is attached
# natively (see dataset_sources) rather than downloaded manually.
import os, sys
REPO_URL = "https://github.com/Malik8122/Cancelable-Multimodal-Biometric-Authentication-for-Critical-Infrastructure.git"
REPO_DIR = "/kaggle/working/repo"
# Voice extends Phase 1's modality set on this branch (not yet merged to
# main/master) - update to the default branch once merged.
REPO_BRANCH = "phase-1-foundation"

if not os.path.isdir(REPO_DIR):
    !git clone -q -b $REPO_BRANCH $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR pull -q

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# h5py is already a project dependency; speechbrain/torchaudio are voice-specific
# and not preinstalled on the base Kaggle image.
!pip install -q h5py speechbrain torchaudio

import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Some hosted GPU sessions intermittently hand out a driver/torch-build
# combination where CUDA reports available but no kernel image exists for
# the actual device - smoke-test with a real op now and fall back to CPU
# rather than crashing deep into training on a broken GPU.
if DEVICE == "cuda":
    try:
        (torch.zeros(1, device=DEVICE) + 1).cpu()
    except Exception as e:  # torch.AcceleratorError, RuntimeError, etc.
        print(f"CUDA smoke test failed ({e}); falling back to CPU.")
        DEVICE = "cpu"
print("Using device:", DEVICE)

Attached natively via `dataset_sources` in `kernel-metadata.json` - no manual download or Kaggle API token needed inside the kernel itself. `models/voice/dataset.py::VoxCelebDataset` expects `root_dir/<speaker_id>/.../*.wav`, so the cell below searches the whole `/kaggle/input` mount for the directory whose immediate children are `idNNNNN` speaker folders, rather than assuming the dataset mounts under a folder matching its Kaggle slug (a real run showed that assumption does not hold for this dataset - see the discovery function's docstring).

In [ ]:
import glob, re

def _discover_dataset_root(search_root: str = "/kaggle/input") -> str:
    """Find the directory whose immediate children are VoxCeleb1 speaker-id
    folders (e.g. id10002), searching the whole /kaggle/input mount rather
    than assuming the dataset mounts under a folder named after its Kaggle
    slug - observed on a real run: the dataset's listed archive paths
    (vox1_indian/content/vox_indian/idNNNNN/...) did not correspond to a
    /kaggle/input/<dataset-slug>/... mount matching that slug at all (zero
    .wav files found there), so the actual mount folder name can't be
    assumed from the dataset's Kaggle URL/slug either."""
    wav_files = glob.glob(f"{search_root}/**/*.wav", recursive=True)
    if not wav_files:
        print(f"No .wav files anywhere under {search_root}. Contents:")
        for root, dirs, files in os.walk(search_root):
            depth = root[len(search_root):].count(os.sep)
            if depth > 3:
                dirs[:] = []
                continue
            print("  " * depth + os.path.basename(root) + "/")
            for f in files[:5]:
                print("  " * (depth + 1) + f)
        raise AssertionError(
            f"No .wav files found anywhere under {search_root} - see the directory "
            "listing printed above, or check the Data tab in the Kaggle UI directly, "
            "and update dataset_sources / this discovery function accordingly."
        )
    relative_parts = os.path.relpath(wav_files[0], search_root).split(os.sep)
    for depth, part in enumerate(relative_parts):
        if re.match(r"^id\d+$", part):
            return os.path.join(search_root, *relative_parts[:depth]) or search_root
    raise AssertionError(
        f"Found .wav files under {search_root} but no ancestor directory matched "
        f"VoxCeleb1's 'idNNNNN' speaker-id naming (checked: {wav_files[0]!r}) - "
        "the dataset's speaker-folder naming may differ from what this project expects; "
        "inspect it via the Data tab and adjust _discover_dataset_root accordingly."
    )

DATASET_ROOT = _discover_dataset_root()
print("Discovered DATASET_ROOT:", DATASET_ROOT)

## 3. Build train/val/test datasets

`VoxCelebDataset` (see `models/voice/dataset.py`) discovers every `.wav` file under `DATASET_ROOT`, maps each top-level directory name to a speaker label, and applies the per-identity 70/15/15 split also used by `notebooks/03_fingerprint_training_and_testing.ipynb` - see `docs/VOICE_MODEL.md` for why this substitutes for VoxCeleb1's official trial-list protocol here. This cell is also this project's actual documentation of dataset composition (spec: "Number of speakers, Number of audio samples, Train/validation/test split") - printed at run time from the real attached data rather than hardcoded, since the exact numbers depend on exactly what Kaggle's copy of this subset contains.

In [ ]:
from models.voice.config import VoiceConfig
from models.voice.dataset import VoxCelebDataset

config = VoiceConfig()  # see models/voice/config.py for every hyperparameter default

train_dataset = VoxCelebDataset(DATASET_ROOT, mode="train", config=config)
val_dataset = VoxCelebDataset(DATASET_ROOT, mode="val", config=config)
test_dataset = VoxCelebDataset(DATASET_ROOT, mode="test", config=config)

print(f"Speakers: {train_dataset.num_speakers}")
print(f"Train utterances: {len(train_dataset)}")
print(f"Val utterances:   {len(val_dataset)}")
print(f"Test utterances:  {len(test_dataset)}")

## 4. Train

Calls `models/voice/train.py::train()` directly - the training loop (AdamW + CosineAnnealingLR + AMP + early stopping) lives there, not duplicated in this notebook, per the project's "avoid notebook-only code inside training modules" convention. Saves `voice_embedder_best.pt`/`_last.pt` during training, then the canonical `voice_embedder.pt` + `.h5` + `training_config.json`.

In [ ]:
from models.voice.train import train

OUTPUT_DIR = f"{REPO_DIR}/models/voice/saved"
CHECKPOINT_PATH = train(DATASET_ROOT, OUTPUT_DIR, config=config)
print("Saved checkpoint:", CHECKPOINT_PATH)

## 5. Evaluate on the held-out test set

Mirrors `evaluation/experiments.py::run_modality_experiment` (used by Face/Iris/Fingerprint) via the voice-specific `evaluation/voice_metrics.py::run_voice_experiment`, which adds precision/recall/F1/confusion-matrix on top of the shared FAR/FRR/EER/ROC-AUC primitives.

In [ ]:
from models.voice.inference import VoiceEmbedder
from evaluation.voice_metrics import run_voice_experiment, save_confusion_matrix_csv, save_roc_csv, save_voice_metrics_csv

fine_tuned_embedder = VoiceEmbedder(checkpoint_path=CHECKPOINT_PATH, device=DEVICE)
assert not fine_tuned_embedder.mock_mode, "Checkpoint failed to load - check the path above."

test_embeddings, test_labels = [], []
for index in range(len(test_dataset)):
    mel_tensor, label = test_dataset[index]
    pseudo_rgb = mel_tensor.unsqueeze(-1).repeat(1, 1, 3).numpy()  # (n_mels, T, 3), matches embeddings/pipelines.py's convention
    test_embeddings.append(fine_tuned_embedder.extract_embedding(pseudo_rgb))
    test_labels.append(str(label))

report = run_voice_experiment(test_embeddings, test_labels)
print(f"EER: {report['eer']:.4f} | Accuracy: {report['accuracy']:.4f} | AUC: {report['auc']:.4f}")
print(f"Precision: {report['precision']:.4f} | Recall: {report['recall']:.4f} | F1: {report['f1']:.4f}")

RESULTS_DIR = f"{REPO_DIR}/evaluation/results"
save_voice_metrics_csv(report, f"{RESULTS_DIR}/voice_metrics.csv")
save_roc_csv(report, f"{RESULTS_DIR}/voice_roc.csv")
save_confusion_matrix_csv(report, f"{RESULTS_DIR}/voice_confusion_matrix.csv")
print("Saved metrics CSVs under", RESULTS_DIR)

## 6. Demo: genuine vs. impostor pair

Same qualitative check the other three modalities' notebooks end with - two utterances from the *same* speaker should score high, two from *different* speakers should score low.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
test_label_names = np.array(test_labels)

def pick_pair(same_speaker: bool):
    for _ in range(200):
        i, j = rng.choice(len(test_embeddings), size=2, replace=False)
        if (test_label_names[i] == test_label_names[j]) == same_speaker:
            return i, j
    raise RuntimeError("Could not find a suitable pair - try a larger test split.")

from evaluation.metrics import cosine_similarity

genuine_i, genuine_j = pick_pair(same_speaker=True)
impostor_i, impostor_j = pick_pair(same_speaker=False)
print("Genuine pair similarity:  ", cosine_similarity(test_embeddings[genuine_i], test_embeddings[genuine_j]))
print("Impostor pair similarity:", cosine_similarity(test_embeddings[impostor_i], test_embeddings[impostor_j]))

## Output

Kaggle automatically persists everything written under `/kaggle/working/` as this kernel's output. After the run finishes, fetch the checkpoints with:
```bash
kaggle kernels output <username>/voice-embedding-training-phase-1-extension -p ./kaggle_output/voice
```
then copy the `.pt`/`.h5` files from `kaggle_output/voice/repo/models/voice/saved/` into this repo's `models/voice/saved/` and commit them (Git LFS picks up the `.pt`/`.h5` files automatically) - or just run `python scripts/run_kaggle_kernels.py --only voice`, which automates all of this.